# Exploração dos dados: receita 2025

In [349]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

df = pd.read_csv("../data/raw/receita_2025.csv", sep=",", decimal=".", encoding="utf-8")

df.columns

Index(['ANO', 'MES', 'COD_MUNICIPIO', 'NOME_MUNICIPIO', 'CD_ORGAO',
       'NOME_ORGAO', 'CD_RECEBIMENTO', 'CD_ORGAO_ORCAMENTARIO',
       'NOME_ORGAO_ORCAMENTARIO', 'CD_UNIDADE_ORCAMENTARIA', 'NOME_UNIDADE',
       'TP_UNIDADE', 'CD_CONTA_SG', 'DS_CONTA_SG', 'NIVEL', 'NR_NIVEL',
       'CD_RECURSO', 'NM_RECURSO', 'CD_FONTE_RECURSO', 'CD_ACOMP_EXEC_ORC',
       'CAR_PECULIAR', 'DS_CAR_PECULIAR', 'VL_ORCADO', 'VL_ARRECADADO'],
      dtype='str')

## Visão geral

Anotações sobre a exploração do arquivo receita_2025.csv:
- Ela tem 24 colunas e 672 linhas
- Algumas colunas, como 'NOME_UNIDADE' e 'TP_UNIDADE', têm apenas 452 linhas preenchidas, enquanto outras têm todas as linhas preenchidas
- 'NM_RECURSO' tem zero linhas preenchidas
- A coluna 'NIVEL' tem apenas dados como 'S' ou 'A'. Sendo 452 linhas com A e 220 linhas com S
- A coluna 'NR_NIVEL' tem apenas números de 1 a 9. E a maioria das linhas tem dados entre 7 e 9
- As linhas com 'NIVEL' = A possuem 'NOME_UNIDADE', enquanto as com 'NIVEL' = S têm 'NOME_UNIDADE' vazio

## NIVEL e NR_NIVEL

In [350]:
filtro_nivel_a = df["NIVEL"] == "A"
filtro_nivel_s = df["NIVEL"] == "S"
filtro_nr_nivel_1 = df["NR_NIVEL"] == 1
filtro_nr_nivel_2 = df["NR_NIVEL"] == 2
filtro_nr_nivel_3 = df["NR_NIVEL"] == 3
filtro_nr_nivel_4 = df["NR_NIVEL"] == 4
filtro_nr_nivel_5 = df["NR_NIVEL"] == 5
filtro_nr_nivel_6 = df["NR_NIVEL"] == 6
filtro_nr_nivel_7 = df["NR_NIVEL"] == 7
filtro_nr_nivel_8 = df["NR_NIVEL"] == 8


df["VL_ARRECADADO"].sum()

np.float64(10455427845.77)

O valor arrecadado de todo o dataframe é R$ 10,455,427,845.77. Mais de seis vezes o valor apenas com NIVEL igual A (R$ 1,549,096,934.03), então somar tudo conta o mesmo dinheiro várias vezes.

In [351]:
df[filtro_nr_nivel_1][["DS_CONTA_SG", "CD_CONTA_SG"]]

,DS_CONTA_SG,CD_CONTA_SG
0,Receitas Correntes,100000000000000
559,Receitas de Capital,200000000000000


In [352]:
print(df[filtro_nr_nivel_1]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_2]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_3]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_4]["VL_ARRECADADO"].sum())

1549096934.03
1549096934.03
1549096934.0300004
1549096934.03


Cada NR_NIVEL, do 1 ao 4, soma exatamente o mesmo valor arrecadado total (R$ 1,549,096,934.03, o mesmo das linhas com NIVEL igual A). Isso provavelmente significa que as linhas se repetem e tem a ver com a decomposição da categoria em níveis.

In [353]:
pd.crosstab(df["NR_NIVEL"], df["NIVEL"])

NIVEL,A,S
NR_NIVEL,,
1,0,2
2,0,9
3,0,20
4,0,39
5,0,67
6,0,24
7,56,54
8,278,5
9,118,0


Do NR_NIVEL 1 ao 6 só existem linhas S. A partir do 7, as linhas A e S se misturam.

Conclusão: NR_NIVEL é o nível da hierarquia. NIVEL A é a ponta da hierarquia, e NIVEL S agrupa as linhas de baixo.

### Teste: a linha S é a soma das linhas A abaixo dela

Pego a linha 3 (NIVEL S) e procuro as linhas A cujo CD_CONTA_SG começa com o mesmo prefixo (o código sem os zeros do fim).

In [354]:
df.loc[3]

ANO                                               2025
MES                                                 12
COD_MUNICIPIO                                      144
NOME_MUNICIPIO                                 PELOTAS
CD_ORGAO                                         54400
NOME_ORGAO                               PM DE PELOTAS
CD_RECEBIMENTO                                  374669
CD_ORGAO_ORCAMENTARIO                                0
NOME_ORGAO_ORCAMENTARIO                            NaN
CD_UNIDADE_ORCAMENTARIA                              0
NOME_UNIDADE                                       NaN
TP_UNIDADE                                         NaN
CD_CONTA_SG                            111200000000000
DS_CONTA_SG                Impostos sobre o Patrimônio
NIVEL                                                S
NR_NIVEL                                             4
CD_RECURSO                                           0
NM_RECURSO                                         NaN
CD_FONTE_R

In [355]:
linha = 3
valor_s = df.loc[linha]["VL_ARRECADADO"]
codigo = df.loc[linha]["CD_CONTA_SG"]
prefixo = str(codigo).rstrip("0")

filtro_prefixo = df["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
df_filhas = df[filtro_prefixo & filtro_nivel_a]

print(prefixo)
print(len(df_filhas))
print(df_filhas["VL_ARRECADADO"].sum(), valor_s)

1112
10
229995095.65 229995095.65


A linha 3 (Impostos sobre o Patrimônio, NIVEL S) arrecadou 229,995,095.65. As 10 linhas A cujo código começa com 1112 somam o mesmo valor. A linha S é o total das linhas A abaixo dela.

## A fazer

- Repetir nos outros três arquivos (receita 2024 e despesa 2024 e 2025)
- Comparar DS_FUNCAO da despesa entre 2024 e 2025
- Valores negativos na receita
- Tipo do CD_CONTA_SG
- VL_EMPENHADO, VL_LIQUIDADO e VL_PAGO na despesa

In [356]:
receita_2024 = pd.read_csv("../data/raw/receita_2024.csv", sep=",", decimal=".", encoding="utf-8")
receita_2025 = pd.read_csv("../data/raw/receita_2025.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2024 = pd.read_csv("../data/raw/despesa_2024.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2025 = pd.read_csv("../data/raw/despesa_2025.csv", sep=",", decimal=".", encoding="utf-8")

arquivos = {
    "receita_2024": receita_2024,
    "receita_2025": receita_2025,
    "despesa_2024": despesa_2024,
    "despesa_2025": despesa_2025,
}

for nome, tabela in arquivos.items():
    print(nome, tabela.shape)

receita_2024 (679, 24)
receita_2025 (672, 24)
despesa_2024 (2990, 41)
despesa_2025 (2894, 41)


In [357]:
list(receita_2024.columns) == list(receita_2025.columns)

True

In [358]:
list(despesa_2024.columns) == list(despesa_2025.columns)

True

In [359]:
def testa_s_contra_a(tabela: pd.DataFrame, linha: int) -> None:
    """Confere se uma linha S é a soma das linhas A abaixo dela."""

    if tabela.loc[linha, "NIVEL"] == "S":

        valor_s = tabela.loc[linha]["VL_ARRECADADO"]
        codigo = tabela.loc[linha]["CD_CONTA_SG"]
        prefixo = str(codigo).rstrip("0")
        filtro_nivel_a = tabela["NIVEL"] == "A"

        filtro_prefixo = tabela["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
        df_filhas = tabela[filtro_prefixo & filtro_nivel_a]
        valor_df_filhas = df_filhas["VL_ARRECADADO"].sum()
        print(f"Codigo: {prefixo}")
        print(f"Total de linhas filhas: {len(df_filhas)}")
        print(f"Total de VL_ARRECADADO das linhas filhas: {valor_df_filhas:,.2f}")
        print(f"VL_ARRECADADO da linha S: {valor_s:,.2f}")


        if valor_s.round(2) == valor_df_filhas.round(2):
            print(f"A linha S é a soma das linhas A abaixo dela.!")
        else:
            print("A linha S NÃO é a soma das linhas A abaixo dela!")
    else:
        print("A linha deve ter NIVEL S")

testa_s_contra_a(receita_2025, 22)
# receita_2025[receita_2025["NIVEL"] == "S"].head(20)

Codigo: 1114
Total de linhas filhas: 5
Total de VL_ARRECADADO das linhas filhas: 140,985,949.00
VL_ARRECADADO da linha S: 140,985,949.00
A linha S é a soma das linhas A abaixo dela.!


In [360]:
def testa_s_contra_a(tabela: pd.DataFrame, linha: int) -> None:
    """Confere se uma linha S é a soma das linhas A abaixo dela."""

    if tabela.loc[linha, "NIVEL"] == "S":

        valor_s = tabela.loc[linha]["VL_ARRECADADO"]
        codigo = tabela.loc[linha]["CD_CONTA_SG"]
        prefixo = str(codigo).rstrip("0")
        filtro_nivel_a = tabela["NIVEL"] == "A"

        filtro_prefixo = tabela["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
        df_filhas = tabela[filtro_prefixo & filtro_nivel_a]
        valor_df_filhas = df_filhas["VL_ARRECADADO"].sum()
        print(f"Codigo: {prefixo}")
        print(f"Total de linhas filhas: {len(df_filhas)}")
        print(f"Total de VL_ARRECADADO das linhas filhas: {valor_df_filhas:,.2f}")
        print(f"VL_ARRECADADO da linha S: {valor_s:,.2f}")


        if valor_s.round(2) == valor_df_filhas.round(2):
            print(f"A linha S é a soma das linhas A abaixo dela.!")
        else:
            print("A linha S NÃO é a soma das linhas A abaixo dela!")
    else:
        print("A linha deve ter NIVEL S")

testa_s_contra_a(receita_2025, 22)

Codigo: 1114
Total de linhas filhas: 5
Total de VL_ARRECADADO das linhas filhas: 140,985,949.00
VL_ARRECADADO da linha S: 140,985,949.00
A linha S é a soma das linhas A abaixo dela.!


In [361]:
df_filtrado = df[filtro_nr_nivel_6]
serie_limpa = df_filtrado["CD_CONTA_SG"].astype(str).str.rstrip("0")
serie_limpa.str.len().mean()
df["caracteres_codigo"] = df["CD_CONTA_SG"].astype(str).str.rstrip("0").str.len()
pd.crosstab(df["NR_NIVEL"], df["caracteres_codigo"])

caracteres_codigo,1,2,3,4,5,6,7,8,10,14,15
NR_NIVEL,,,,,,,,,,,
1,2,0,0,0,0,0,0,0,0,0,0
2,0,9,0,0,0,0,0,0,0,0,0
3,0,0,20,0,0,0,0,0,0,0,0
4,0,0,0,39,0,0,0,0,0,0,0
5,0,0,0,0,15,52,0,0,0,0,0
6,0,0,0,0,0,0,24,0,0,0,0
7,0,0,0,0,0,0,0,54,0,0,56
8,0,0,0,0,0,0,0,0,5,0,278
9,0,0,0,0,0,0,0,0,0,5,113


In [362]:
df_filtrado = df[filtro_nivel_a]
df_filtrado.groupby(["NR_NIVEL", "NIVEL", "DS_CONTA_SG", "CD_CONTA_SG"])["VL_ARRECADADO"].sum().sample(20)

NR_NIVEL  NIVEL  DS_CONTA_SG                                                                                                       CD_CONTA_SG    
8         A      Multas Previstas em Legislação Específica - Art. 7º Lei 4.421 - Vigilância Sanitária - Multas e Juros             191101020300001        5,122.25
                 Contrato 78.458 - BRDE                                                                                            211201010400001    3,472,506.17
                 Rede Bem Cuidar e UBS - Qualifica Ações Puericultura - Portaria SES nº 1.228/25                                   242150010700001       50,000.00
                 Incremento MAC - Portaria nº 7.446/2025 - Hospital Beneficência Portuguesa                                        171350213200001    2,000,000.00
                 Alienação de Bens Móveis e Semoventes - Exceto RPPS                                                               221301010200002      193,350.00
                 Fortalecimento do Sis

In [363]:
df["CD_CONTA_SG"] = df["CD_CONTA_SG"].astype(str)
codigo = "111250"
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df_filtrado_codigo = df[(filtro_codigo) & (filtro_nivel_a)]
# df_filtrado_codigo[["NOME_ORGAO_ORCAMENTARIO", "NIVEL", "NR_NIVEL", "DS_CONTA_SG", "CD_CONTA_SG", "VL_ARRECADADO"]].iloc[0]
df_filtrado_codigo.iloc[0]


ANO                                                                     2025
MES                                                                       12
COD_MUNICIPIO                                                            144
NOME_MUNICIPIO                                                       PELOTAS
CD_ORGAO                                                               54400
NOME_ORGAO                                                     PM DE PELOTAS
CD_RECEBIMENTO                                                        374669
CD_ORGAO_ORCAMENTARIO                                                     40
NOME_ORGAO_ORCAMENTARIO                      Secretaria Municipal da Fazenda
CD_UNIDADE_ORCAMENTARIA                                                   44
NOME_UNIDADE                                 Secretaria Municipal da Fazenda
TP_UNIDADE                                                               PRM
CD_CONTA_SG                                                  111250010000001

In [364]:
# Nível 1
codigo_folha = "111250010000001"
codigo = codigo_folha[:1]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_1) & (filtro_codigo)].iloc[0]


ANO                                      2025
MES                                        12
COD_MUNICIPIO                             144
NOME_MUNICIPIO                        PELOTAS
CD_ORGAO                                54400
NOME_ORGAO                      PM DE PELOTAS
CD_RECEBIMENTO                         374669
CD_ORGAO_ORCAMENTARIO                       0
NOME_ORGAO_ORCAMENTARIO                   NaN
CD_UNIDADE_ORCAMENTARIA                     0
NOME_UNIDADE                              NaN
TP_UNIDADE                                NaN
CD_CONTA_SG                   100000000000000
DS_CONTA_SG                Receitas Correntes
NIVEL                                       S
NR_NIVEL                                    1
CD_RECURSO                                  0
NM_RECURSO                                NaN
CD_FONTE_RECURSO                            0
CD_ACOMP_EXEC_ORC                           0
CAR_PECULIAR                                0
DS_CAR_PECULIAR                 Nã

In [365]:
# Nível 2
codigo = codigo_folha[:2]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_2) & (filtro_codigo)].iloc[0]

ANO                                                               2025
MES                                                                 12
COD_MUNICIPIO                                                      144
NOME_MUNICIPIO                                                 PELOTAS
CD_ORGAO                                                         54400
NOME_ORGAO                                               PM DE PELOTAS
CD_RECEBIMENTO                                                  374669
CD_ORGAO_ORCAMENTARIO                                                0
NOME_ORGAO_ORCAMENTARIO                                            NaN
CD_UNIDADE_ORCAMENTARIA                                              0
NOME_UNIDADE                                                       NaN
TP_UNIDADE                                                         NaN
CD_CONTA_SG                                            110000000000000
DS_CONTA_SG                Impostos, Taxas e Contribuições de Melhoria
NIVEL 

In [366]:
# Nível 3
codigo = codigo_folha[:3]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_3) & (filtro_codigo)].iloc[0]

ANO                                   2025
MES                                     12
COD_MUNICIPIO                          144
NOME_MUNICIPIO                     PELOTAS
CD_ORGAO                             54400
NOME_ORGAO                   PM DE PELOTAS
CD_RECEBIMENTO                      374669
CD_ORGAO_ORCAMENTARIO                    0
NOME_ORGAO_ORCAMENTARIO                NaN
CD_UNIDADE_ORCAMENTARIA                  0
NOME_UNIDADE                           NaN
TP_UNIDADE                             NaN
CD_CONTA_SG                111000000000000
DS_CONTA_SG                       Impostos
NIVEL                                    S
NR_NIVEL                                 3
CD_RECURSO                               0
NM_RECURSO                             NaN
CD_FONTE_RECURSO                         0
CD_ACOMP_EXEC_ORC                        0
CAR_PECULIAR                             0
DS_CAR_PECULIAR              Não se Aplica
VL_ORCADO                   375,126,608.98
VL_ARRECADA

In [367]:
# Nível 4
codigo = codigo_folha[:4]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_4) & (filtro_codigo)].iloc[0]

ANO                                               2025
MES                                                 12
COD_MUNICIPIO                                      144
NOME_MUNICIPIO                                 PELOTAS
CD_ORGAO                                         54400
NOME_ORGAO                               PM DE PELOTAS
CD_RECEBIMENTO                                  374669
CD_ORGAO_ORCAMENTARIO                                0
NOME_ORGAO_ORCAMENTARIO                            NaN
CD_UNIDADE_ORCAMENTARIA                              0
NOME_UNIDADE                                       NaN
TP_UNIDADE                                         NaN
CD_CONTA_SG                            111200000000000
DS_CONTA_SG                Impostos sobre o Patrimônio
NIVEL                                                S
NR_NIVEL                                             4
CD_RECURSO                                           0
NM_RECURSO                                         NaN
CD_FONTE_R

In [368]:
# Nível 5
codigo = codigo_folha[:6]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_5) & (filtro_codigo)]

,ANO,MES,COD_MUNICIPIO,NOME_MUNICIPIO,CD_ORGAO,NOME_ORGAO,CD_RECEBIMENTO,CD_ORGAO_ORCAMENTARIO,NOME_ORGAO_ORCAMENTARIO,CD_UNIDADE_ORCAMENTARIA,...,NR_NIVEL,CD_RECURSO,NM_RECURSO,CD_FONTE_RECURSO,CD_ACOMP_EXEC_ORC,CAR_PECULIAR,DS_CAR_PECULIAR,VL_ORCADO,VL_ARRECADADO,caracteres_codigo
4,2025,12,144,PELOTAS,54400,PM DE PELOTAS,374669,0,NaN,0,...,5,0,NaN,0,0,0,Não se Aplica,"164,362,337.22","184,095,028.53",5


In [369]:
# Nível 6
codigo = codigo_folha[:7]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_6) & (filtro_codigo)]

,ANO,MES,COD_MUNICIPIO,NOME_MUNICIPIO,CD_ORGAO,NOME_ORGAO,CD_RECEBIMENTO,CD_ORGAO_ORCAMENTARIO,NOME_ORGAO_ORCAMENTARIO,CD_UNIDADE_ORCAMENTARIA,...,NR_NIVEL,CD_RECURSO,NM_RECURSO,CD_FONTE_RECURSO,CD_ACOMP_EXEC_ORC,CAR_PECULIAR,DS_CAR_PECULIAR,VL_ORCADO,VL_ARRECADADO,caracteres_codigo


In [380]:
# Nível a
codigo = codigo_folha[:6]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
print(df[(filtro_nivel_a) & (filtro_codigo)][["VL_ARRECADADO", "NIVEL", "NR_NIVEL"]])
print(df[(filtro_nivel_a) & (filtro_codigo)]["VL_ARRECADADO"].sum().round(2))

    VL_ARRECADADO NIVEL  NR_NIVEL
5  148,422,688.20     A         7
6            0.06     A         7
7            0.00     A         7
8    2,575,923.48     A         7
9   22,059,179.86     A         7
10       1,398.00     A         7
11  11,035,838.93     A         7
184095028.53


## O que está comprovado
- A receita é uma hierarquia de categorias e subcategorias. O NR_NIVEL é a profundidade, e o CD_CONTA_SG mostra a ligação: o código do filho começa com o código do pai (o ramo do IPTU, do nível 1 ao 5, comprova).
- NIVEL = S agrupa, NIVEL = A é a ponta. A linha S é a soma das linhas A abaixo dela (confirmado em linhas que você testou, e na verificação completa que fiz: 232 de 238 em 2024 e 216 de 220 em 2025).
- O que cada NIVEL significa. A (analítica) é a ponta da hierarquia: é onde o valor é registrado, e as linhas A só aparecem do NR_NIVEL 7 ao 9. S (sintética) é uma categoria que agrupa as de baixo, e vai do NR_NIVEL 1 ao 8. Exemplo: a linha S do IPTU (nível 5) arrecadou R$ 184.095.028,53, e as 7 linhas A sob ela somam exatamente esse valor.
- Somar a tabela inteira conta o mesmo dinheiro várias vezes. Para o total real, some só NIVEL == "A". Isso vai direto para a Etapa 5.
- O tamanho do prefixo depende do nível (1, 2, 3, 4, 6, 7, 8, 10 caracteres), e o rstrip("0") erra no nível 5, por cortar um zero legítimo.
- A hierarquia pula níveis. O IPTU vai do nível 5 direto para a ponta no nível 7.

# Receitas 2024
Fazer a exploração com o arquivo receitas 2024.

In [381]:
receita_2024 = pd.read_csv("../data/raw/receita_2024.csv", sep=",", decimal=".", encoding="utf-8")

receita_2024.shape

(679, 24)

In [382]:
receita_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 679 entries, 0 to 678
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ANO                      679 non-null    int64  
 1   MES                      679 non-null    int64  
 2   COD_MUNICIPIO            679 non-null    int64  
 3   NOME_MUNICIPIO           679 non-null    str    
 4   CD_ORGAO                 679 non-null    int64  
 5   NOME_ORGAO               679 non-null    str    
 6   CD_RECEBIMENTO           679 non-null    int64  
 7   CD_ORGAO_ORCAMENTARIO    679 non-null    int64  
 8   NOME_ORGAO_ORCAMENTARIO  441 non-null    str    
 9   CD_UNIDADE_ORCAMENTARIA  679 non-null    int64  
 10  NOME_UNIDADE             441 non-null    str    
 11  TP_UNIDADE               441 non-null    str    
 12  CD_CONTA_SG              679 non-null    int64  
 13  DS_CONTA_SG              679 non-null    str    
 14  NIVEL                    679 non-null

In [383]:
receita_2024.value_counts("NIVEL")

NIVEL
A    441
S    238
Name: count, dtype: int64

In [385]:
receita_2024["VL_ARRECADADO"].sum().round(2)

np.float64(10044803801.06)

In [387]:
filtro_nivel_a = receita_2024["NIVEL"] == "A"
filtro_nivel_s = receita_2024["NIVEL"] == "S"
filtro_nr_nivel_1 = receita_2024["NR_NIVEL"] == 1
filtro_nr_nivel_2 = receita_2024["NR_NIVEL"] == 2
filtro_nr_nivel_3 = receita_2024["NR_NIVEL"] == 3
filtro_nr_nivel_4 = receita_2024["NR_NIVEL"] == 4
filtro_nr_nivel_5 = receita_2024["NR_NIVEL"] == 5
filtro_nr_nivel_6 = receita_2024["NR_NIVEL"] == 6
filtro_nr_nivel_7 = receita_2024["NR_NIVEL"] == 7
filtro_nr_nivel_8 = receita_2024["NR_NIVEL"] == 8

In [389]:
receita_2024[filtro_nivel_a]["VL_ARRECADADO"].sum().round(2)

np.float64(1458681677.02)

In [390]:
receita_2024[filtro_nr_nivel_1]["VL_ARRECADADO"].sum().round(2)

np.float64(1458681677.02)

In [391]:
receita_2024[filtro_nivel_a]["VL_ARRECADADO"].sum().round(2)

np.float64(1458681677.02)

In [392]:
pd.crosstab(receita_2024["NR_NIVEL"], receita_2024["NIVEL"])

NIVEL,A,S
NR_NIVEL,,
1,0,3
2,0,9
3,0,21
4,0,40
5,0,75
6,0,25
7,51,59
8,285,6
9,105,0


In [398]:
receita_2024[filtro_nr_nivel_1][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
0,100000000000000,Receitas Correntes,S,1,"1,472,878,245.72"
543,200000000000000,Receitas de Capital,S,1,"59,254,414.61"
665,9000000000000000,(R) Deduções da Receita Corrente,S,1,"-73,450,983.31"


In [394]:
receita_2024["caracteres_codigo"] = receita_2024["CD_CONTA_SG"].astype(str).str.rstrip("0").str.len()
pd.crosstab(receita_2024["NR_NIVEL"], receita_2024["caracteres_codigo"])

caracteres_codigo,1,2,3,4,5,6,7,8,10,14,15,16
NR_NIVEL,,,,,,,,,,,,
1,3,0,0,0,0,0,0,0,0,0,0,0
2,0,9,0,0,0,0,0,0,0,0,0,0
3,0,0,21,0,0,0,0,0,0,0,0,0
4,0,0,0,40,0,0,0,0,0,0,0,0
5,0,0,0,0,17,58,0,0,0,0,0,0
6,0,0,0,0,0,0,25,0,0,0,0,0
7,0,0,0,0,0,0,0,59,0,0,51,0
8,0,0,0,0,0,0,0,0,6,0,275,10
9,0,0,0,0,0,0,0,0,0,5,100,0


In [399]:
filtro_caracteres_codigo = receita_2024["caracteres_codigo"] == 16
receita_2024[filtro_caracteres_codigo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
668,9111250010000101,(R) Deduções - IPTU - Principal - Renúncia Fiscal,A,8,0.00
669,9111253010000101,(R) Deduções - ITBI - Principal Renúncia Fiscal,A,8,0.00
670,9111451110000101,(R) Deduções - ISS - Principal - Renúncia Fiscal,A,8,0.00
671,9112201010000101,(R) Deduções - Taxas pela Prestação de Serviço...,A,8,0.00
673,9171151110000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-26,429,839.66"
674,9171152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-265,020.72"
675,9172150010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-30,793,411.64"
676,9172151010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-15,157,400.20"
677,9172152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-389,407.12"
678,9172953010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-415,903.97"


In [402]:
receita_2024[filtro_nivel_s][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]].sample(3)

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
192,171000000000000,Transferências da União e de suas Entidades,S,3,"488,160,843.26"
78,131102000000000,"Concessão, Permissão, Autorização ou Cessão do...",S,5,"2,070,159.60"
154,132101010200000,Remuneração de Depósitos de Recursos Não Vincu...,S,8,"1,362,586.78"


In [403]:
testa_s_contra_a(receita_2024, 192)
testa_s_contra_a(receita_2024, 78)
testa_s_contra_a(receita_2024, 154)

Codigo: 171
Total de linhas filhas: 104
Total de VL_ARRECADADO das linhas filhas: 488,160,843.26
VL_ARRECADADO da linha S: 488,160,843.26
A linha S é a soma das linhas A abaixo dela.!
Codigo: 131102
Total de linhas filhas: 2
Total de VL_ARRECADADO das linhas filhas: 2,070,159.60
VL_ARRECADADO da linha S: 2,070,159.60
A linha S é a soma das linhas A abaixo dela.!
Codigo: 1321010102
Total de linhas filhas: 11
Total de VL_ARRECADADO das linhas filhas: 1,362,586.78
VL_ARRECADADO da linha S: 1,362,586.78
A linha S é a soma das linhas A abaixo dela.!


## Conclusões da receita 2024

- A estrutura é a mesma de 2025. As linhas A ficam do NR_NIVEL 7 ao 9 e as S do 1 ao 8. Somar a tabela toda dá R$ 10,04 bi, quase sete vezes o valor das linhas A (R$ 1.458.681.677,02), então somar tudo conta o mesmo dinheiro várias vezes. O NR_NIVEL 1 soma exatamente o mesmo que as linhas A.
- Os prefixos têm os mesmos tamanhos por nível de 2025, e o nível 5 também se divide em 5 e 6 caracteres.
**Diferenças em relação a 2025**

- Em 2024 existe uma terceira linha no NR_NIVEL 1: "(R) Deduções da Receita Corrente" (código 9000000000000000, R$ -73.450.983,31). Em 2025 há só duas.
- Os 14 códigos que começam com 9 pertencem às deduções e têm 16 dígitos, em vez de 15 (renúncia fiscal de IPTU, ITBI, ISS e taxas, e deduções para formação do FUNDEB).
- O total das linhas A de 2024 já inclui essas deduções. Sem elas seria R$ 1.532.132.660,33.

**Problemas para a pipeline resolver**

- O `CD_CONTA_SG` deve ser lido como texto, porque o código tem 15 ou 16 dígitos.
- Decidir se a receita arrecadada considera as deduções (líquida) ou não (bruta).

In [412]:
filtro_negativo = receita_2024["VL_ARRECADADO"] < 0
receita_2024[filtro_negativo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
665,9000000000000000,(R) Deduções da Receita Corrente,S,1,"-73,450,983.31"
666,9100000000000000,(R) Deduções da Receita Corrente,S,2,"-73,450,983.31"
672,9170000000000000,(R) Deduções da Receita Corrente,S,3,"-73,450,983.31"
673,9171151110000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-26,429,839.66"
674,9171152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-265,020.72"
675,9172150010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-30,793,411.64"
676,9172151010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-15,157,400.20"
677,9172152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-389,407.12"
678,9172953010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-415,903.97"


In [413]:
filtro_negativo = receita_2025["VL_ARRECADADO"] < 0
receita_2025[filtro_negativo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
216,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"-28,695,886.97"
220,171152010000001,Cota-Parte do Imposto Sobre a Propriedade Terr...,A,7,"-266,813.13"
367,172150010000001,Cota-Parte do ICMS - Principal,A,7,"-33,430,718.82"
370,172151010000001,Cota-Parte do IPVA - Principal,A,7,"-16,166,711.13"
373,172152010000001,Cota-Parte do IPI - Municípios - Principal,A,7,"-390,540.13"


## Valores negativos na receita

Há 9 linhas com `VL_ARRECADADO` negativo em 2024 (6 A e 3 S) e 5 linhas em 2025 (todas A). Não são erro de preenchimento: são deduções da receita. O município recebe repasses como FPM, ITR, ICMS, IPVA e IPI, e 20% deles são retidos para formar o FUNDEB, o fundo da educação básica. A contabilidade registra essa retenção como valor negativo, que reduz a receita arrecadada.

- Em 2024, as deduções ficam num ramo separado, com códigos que começam com 9 e descrição "(R) Dedução...". Somam -R$ 73.450.983,31.
- Em 2025, os mesmos cinco repasses aparecem negativos, com valores parecidos, mas com o código da própria conta de receita (por exemplo, `171151110000001`, Cota-Parte do FPM) e sem a palavra "dedução" na descrição. Somam -R$ 78.950.670,18.

**Conclusões**

- Os negativos são válidos. A pipeline não deve descartá-los nem trocar o sinal.
- O total das linhas A é receita líquida nos dois anos, porque as deduções entram na soma em 2024 e em 2025.
- Comparar categorias entre os anos exige cuidado: em 2024 o desconto do FUNDEB está num ramo à parte, e em 2025 ele está misturado nas contas dos repasses.

In [414]:
filtro_171151110000001 = receita_2025["CD_CONTA_SG"] == 171151110000001
receita_2025[filtro_171151110000001][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
215,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"143,456,215.81"
216,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"-28,695,886.97"
